# Adapter
  
Dieses Notebook trainiert einen Adapter auf den bestehenden CLIP Embeddings
  


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from scipy.spatial import cKDTree
import geopandas as gpd




def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())

PROJECT_ROOT = find_project_root()
METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
ADAPTER = CFG["vpr"].get("adapter", "none")
METHOD_DIR = EMBEDDING_DIR / f"{METHOD}"
POSITIVE_PATH = PROJECT_ROOT / "data" / "processed" / "train_positive_candidates.csv"

SEED = int(CFG["vpr"].get("split_seed", 42))
ADAPTER_CFG = CFG["vpr"]["adapter_training"]
BATCH_SIZE = ADAPTER_CFG["batch_size"]
EPOCHS = ADAPTER_CFG["epochs"]  
LEARNING_RATE = ADAPTER_CFG["learning_rate"]
MARGIN = ADAPTER_CFG["margin"]
UNCERTAIN_RADIUS_M = CFG["vpr"]["uncertain_radius_m"]
HARD_NEGATIVE_MIN_M = CFG["vpr"]["hard_negative_min_m"]
HARD_NEGATIVE_MAX_M = CFG["vpr"]["hard_negative_max_m"]
POSITIVE_RADIUS_M = CFG["vpr"]["positive_radius_m"]
HARD_NEGATIVE_PROBABILITY = ADAPTER_CFG["hard_negative_probability"]


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.adapter import LinearAdapter

if ADAPTER == "none" or ADAPTER == "None":
    EMBEDDING_NAME = METHOD
else:
    EMBEDDING_NAME = f"{METHOD}_{ADAPTER}"


if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"


MODEL_DIR = PROJECT_ROOT / "models" / "adapters"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_PATH = MODEL_DIR / f"{METHOD}_linear.pt"

positive_candidates = pd.read_csv(POSITIVE_PATH)


print(positive_candidates["distance_m"].describe())
print()
print(positive_candidates["distance_m"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Project:             {PROJECT_ROOT}")
print(f"Method:              {METHOD}")
print(f"Model:               {MODEL_ID}")
print(f"Device:              {DEVICE}")
print(f"Batch size:          {BATCH_SIZE}")
print(f"Epochs:              {EPOCHS}")
print(f"Learning rate:       {LEARNING_RATE}")
print(f"Margin:              {MARGIN}")
print(f"Positive radius:     {POSITIVE_RADIUS_M} m")
print(f"Hard negatives:      {HARD_NEGATIVE_MIN_M}-{HARD_NEGATIVE_MAX_M} m")
print(f"Uncertain radius:    {UNCERTAIN_RADIUS_M} m  (aus dem Training ausgeschlossen)")
print(f"Adapter output:      {ADAPTER_PATH}")


# CLIP Embeddings laden
  
Baseline CLIP Embeddings kein trainierete Adapter

In [ ]:


EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD

embedding_path = EMBEDDING_DIR / f"{METHOD}_embeddings.npy"

metadata_path = EMBEDDING_DIR / f"{METHOD}_metadata.parquet"


embeddings = np.load(embedding_path)

embedding_metadata = pd.read_parquet(metadata_path)



if len(embeddings) != len(embedding_metadata):
    raise ValueError(
        f"Embedding/Metadata-Mismatch: "
        f"{len(embeddings)} Embeddings vs. "
        f"{len(embedding_metadata)} Metadata-Zeilen"
    )


print(f"Embeddings: {embeddings.shape}")

print(f"Metadata:   {embedding_metadata.shape}")


In [ ]:
import random

# ------------------------------------------------------------
# fit / val Split -- ausschliesslich innerhalb von "train".
# database und query werden in diesem Notebook NIE angefasst.
# ------------------------------------------------------------

train_mask_all = (embedding_metadata["split"] == "train").to_numpy()

if not train_mask_all.any():
    raise RuntimeError(
        "Keine train-Embeddings gefunden. 04_embeddings muss mit "
        '["train", "database", "query"] gelaufen sein.'
    )

seq_col = embedding_metadata["sequence_id"].astype(str)

# Aufteilen auf SEQUENZ-Ebene, nicht auf Bildebene: aufeinanderfolgende
# Frames derselben Fahrt duerfen nicht in fit UND val landen.
rng = random.Random(SEED)
train_seqs = sorted(seq_col[train_mask_all].unique())
rng.shuffle(train_seqs)

n_val = max(1, int(0.1 * len(train_seqs)))
val_seqs = set(train_seqs[:n_val])
fit_seqs = set(train_seqs[n_val:])

fit_mask = train_mask_all & seq_col.isin(fit_seqs).to_numpy()
val_mask = train_mask_all & seq_col.isin(val_seqs).to_numpy()

fit_embeddings = embeddings[fit_mask]
val_embeddings = embeddings[val_mask]
fit_metadata = embedding_metadata[fit_mask].reset_index(drop=True)
val_metadata = embedding_metadata[val_mask].reset_index(drop=True)

assert not (fit_seqs & val_seqs), "Sequenz-Leakage zwischen fit und val!"

print(f"Train-Sequenzen gesamt: {len(train_seqs):,}")
print(f"  fit: {len(fit_seqs):,} Sequenzen / {fit_mask.sum():,} Bilder")
print(f"  val: {len(val_seqs):,} Sequenzen / {val_mask.sum():,} Bilder")


In [ ]:
def to_metric_xy(lat, lon, crs=None):
    """Lat/Lon -> Meter. Gibt das CRS zurueck, damit alle dasselbe benutzen."""
    gs = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326")
    if crs is None:
        crs = gs.estimate_utm_crs()
    gs = gs.to_crs(crs)
    return np.c_[gs.x.to_numpy(), gs.y.to_numpy()], crs


In [ ]:
# ------------------------------------------------------------
# Index-Tabelle fuer den fit-Teil
# ------------------------------------------------------------

fit_id_to_index = {
    image_id: index for index, image_id in enumerate(fit_metadata["image_id"])
}

print(f"fit-Bilder: {len(fit_metadata):,}")


# ------------------------------------------------------------
# Mini-Retrieval auf val, als ehrliche Stoppmetrik.
#
# Die val-Sequenzen werden in zwei Haelften geteilt: eine dient als
# Mini-Database, die andere als Mini-Query. Gemessen wird Recall@1
# gegen denselben Radius wie in der spaeteren Evaluation.
# ------------------------------------------------------------

val_seq_list = sorted(val_seqs)
half = max(1, len(val_seq_list) // 2)
val_db_seqs = set(val_seq_list[:half])
val_q_seqs = set(val_seq_list[half:]) or val_db_seqs

val_seq_col = val_metadata["sequence_id"].astype(str)
val_db_mask = val_seq_col.isin(val_db_seqs).to_numpy()
val_q_mask = val_seq_col.isin(val_q_seqs).to_numpy()

val_db_embeddings = val_embeddings[val_db_mask]
val_q_embeddings = val_embeddings[val_q_mask]
val_db_metadata = val_metadata[val_db_mask].reset_index(drop=True)
val_q_metadata = val_metadata[val_q_mask].reset_index(drop=True)

# Vorabpruefung: haben die val-Queries ueberhaupt Abdeckung?
_vdb, _crs = to_metric_xy(
    val_db_metadata["lat"].to_numpy(), val_db_metadata["lon"].to_numpy()
)
_vq, _ = to_metric_xy(
    val_q_metadata["lat"].to_numpy(), val_q_metadata["lon"].to_numpy(), crs=_crs
)
_n_cov = sum(
    len(n) > 0 for n in cKDTree(_vdb).query_ball_point(_vq, r=UNCERTAIN_RADIUS_M)
)
print(f"  val-Queries mit Abdeckung: {_n_cov:,} von {len(_vq):,}")
if _n_cov == 0:
    raise RuntimeError(
        "Keine val-Query hat eine Database-Abdeckung -- Early Stopping waere "
        "wirkungslos. val-Anteil erhoehen (n_val) oder Radius pruefen."
    )

print(f"  val-database: {len(val_db_metadata):,} Bilder")
print(f"  val-query:    {len(val_q_metadata):,} Bilder")


# Positive Candidate Laden

In [ ]:
# ------------------------------------------------------------
# Train-Positives (Paare INNERHALB von train, aus 01 erzeugt)
# ------------------------------------------------------------

positive_candidates = pd.read_csv(POSITIVE_PATH)

expected = {"anchor_image_id", "positive_image_id", "distance_m"}
missing = expected - set(positive_candidates.columns)
if missing:
    raise ValueError(
        f"{POSITIVE_PATH.name} fehlen Spalten: {sorted(missing)}. "
        "Die Zelle 'Train-Positives' in 01_mapillary_coverage muss gelaufen sein."
    )

print(f"Train-Positive-Paare: {len(positive_candidates):,}")
print(positive_candidates["distance_m"].describe())


# IDs in Embedding Indizes umwandeln

In [ ]:
# ------------------------------------------------------------
# image_id -> Index im fit-Embedding-Array
#
# Paare, bei denen ein Partner in val liegt, werden verworfen --
# sonst waere der Val-Split wertlos.
# ------------------------------------------------------------

positive_pairs = []
dropped = 0

for row in positive_candidates.itertuples(index=False):
    anchor_index = fit_id_to_index.get(row.anchor_image_id)
    positive_index = fit_id_to_index.get(row.positive_image_id)

    if anchor_index is None or positive_index is None:
        dropped += 1
        continue

    positive_pairs.append((anchor_index, positive_index, row.distance_m))

if not positive_pairs:
    raise RuntimeError("Keine Positive-Paare im fit-Split gefunden.")

print(f"Positive Pairs (fit): {len(positive_pairs):,}")
print(f"Verworfen (val oder nicht embedded): {dropped:,}")


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6_371_000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    diff_lat = lat2 - lat1
    diff_lon = lon2 - lon1

    a = (
        np.sin(diff_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(diff_lon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius * c




In [ ]:
class TripletDataset(Dataset):
    """
    Anchor, Positive und Negative kommen ALLE aus dem fit-Teil von train.
    database und query werden hier nie beruehrt.

    Bandaufteilung (alles relativ zum Anchor):
        <= positive_radius_m                        Positive
        positive_radius_m .. uncertain_radius_m     Uncertain -> ausgeschlossen
        hard_negative_min_m .. hard_negative_max_m  Hard Negative
        > uncertain_radius_m                        Easy Negative
    """

    def __init__(
        self,
        fit_embeddings,
        positive_pairs,
        fit_metadata,
        positive_radius_m,
        uncertain_radius_m,
        hard_negative_min_m,
        hard_negative_max_m,
        hard_negative_probability=0.75,
    ):
        if not (positive_radius_m <= uncertain_radius_m <= hard_negative_min_m):
            raise ValueError(
                "Es muss gelten: positive <= uncertain <= hard_negative_min. "
                "Sonst ueberlappen Hard Negatives mit der Ground Truth."
            )

        self.fit_embeddings = fit_embeddings
        self.positive_pairs = positive_pairs
        self.fit_metadata = fit_metadata
        self.hard_negative_probability = hard_negative_probability

        # Positive Indizes je Anchor
        self.positive_by_anchor = {}
        for anchor_index, positive_index, _ in positive_pairs:
            self.positive_by_anchor.setdefault(anchor_index, set()).add(positive_index)

        # --- einmalig vorberechnen statt bei jedem __getitem__ -------------
        xy, _ = to_metric_xy(
            fit_metadata["lat"].to_numpy(),
            fit_metadata["lon"].to_numpy(),
        )

        self.n_fit = len(xy)
        tree = cKDTree(xy)

        anchors = sorted(self.positive_by_anchor.keys())
        pts = xy[anchors]

        # query_ball_point nimmt ein Array -> ein Aufruf statt N.
        near_uncertain = tree.query_ball_point(pts, r=uncertain_radius_m)
        near_hard_max = tree.query_ball_point(pts, r=hard_negative_max_m)
        near_hard_min = tree.query_ball_point(pts, r=hard_negative_min_m)

        self.hard_by_anchor = {}
        self.blocked_by_anchor = {}

        for slot, ai in enumerate(anchors):
            uncertain_set = set(near_uncertain[slot])
            uncertain_set |= self.positive_by_anchor[ai]
            uncertain_set.add(ai)

            hard_set = set(near_hard_max[slot]) - set(near_hard_min[slot])

            self.hard_by_anchor[ai] = np.fromiter(
                hard_set, dtype=np.int64, count=len(hard_set)
            )
            # Alles <= uncertain_radius ist als Negativ gesperrt.
            self.blocked_by_anchor[ai] = uncertain_set

        n_hard = np.mean([len(v) for v in self.hard_by_anchor.values()])
        print(f"Nachbarschaften vorberechnet fuer {len(anchors):,} Anchor-Bilder")
        print(f"Hard Negatives je Anchor (Mittel): {n_hard:.1f}")

    def __len__(self):
        return len(self.positive_pairs)

    def _sample_negative(self, anchor_index):
        hard = self.hard_by_anchor.get(anchor_index)
        blocked = self.blocked_by_anchor.get(anchor_index, frozenset())

        if hard is not None and len(hard):
            if np.random.random() < self.hard_negative_probability:
                return int(hard[np.random.randint(len(hard))])

        # Easy Negative per Rejection Sampling: gesperrt sind nur ein paar
        # Dutzend von n_fit Kandidaten, der erste Wurf sitzt fast immer.
        for _ in range(32):
            candidate = int(np.random.randint(self.n_fit))
            if candidate not in blocked:
                return candidate

        if hard is not None and len(hard):
            return int(hard[np.random.randint(len(hard))])

        raise RuntimeError(f"Keine negativen Kandidaten fuer Anchor {anchor_index}.")

    def __getitem__(self, index):
        anchor_index, positive_index, _ = self.positive_pairs[index]

        negative_index = self._sample_negative(anchor_index)

        return (
            torch.from_numpy(self.fit_embeddings[anchor_index]).float(),
            torch.from_numpy(self.fit_embeddings[positive_index]).float(),
            torch.from_numpy(self.fit_embeddings[negative_index]).float(),
        )


In [ ]:
# ------------------------------------------------------------
# Dataset + DataLoader
# ------------------------------------------------------------

dataset = TripletDataset(
    fit_embeddings=fit_embeddings,
    positive_pairs=positive_pairs,
    fit_metadata=fit_metadata,
    positive_radius_m=POSITIVE_RADIUS_M,
    uncertain_radius_m=UNCERTAIN_RADIUS_M,
    hard_negative_min_m=HARD_NEGATIVE_MIN_M,
    hard_negative_max_m=HARD_NEGATIVE_MAX_M,
    hard_negative_probability=HARD_NEGATIVE_PROBABILITY,
)

print(f"Dataset size: {len(dataset):,}")

MAX_PAIRS_PER_EPOCH = 200_000
if len(dataset) > MAX_PAIRS_PER_EPOCH:
    from torch.utils.data import RandomSampler

    sampler = RandomSampler(dataset, replacement=False, num_samples=MAX_PAIRS_PER_EPOCH)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler)
    print(
        f"Deckel aktiv: {MAX_PAIRS_PER_EPOCH:,} von {len(dataset):,} Paaren je Epoche"
    )
else:
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

anchor, positive, negative = next(iter(train_loader))
print("Anchor:  ", anchor.shape)
print("Positive:", positive.shape)
print("Negative:", negative.shape)


# ------------------------------------------------------------
# Sanity Check: liegt das gezogene Negative wirklich ausserhalb
# der Uncertain-Zone?
# ------------------------------------------------------------

anchor_index, positive_index, _ = positive_pairs[0]
anchor_row = fit_metadata.iloc[anchor_index]

checked = [dataset._sample_negative(anchor_index) for _ in range(200)]
d = haversine_distance(
    anchor_row["lat"],
    anchor_row["lon"],
    fit_metadata["lat"].to_numpy()[checked],
    fit_metadata["lon"].to_numpy()[checked],
)

print()
print("200 gezogene Negatives, Abstand zum Anchor:")
print(f"  Minimum: {d.min():8.1f} m   (muss > {UNCERTAIN_RADIUS_M} m sein)")
print(f"  Median:  {np.median(d):8.1f} m")
assert d.min() > UNCERTAIN_RADIUS_M, "Negative aus der Uncertain-Zone gezogen!"


# Adapter erstellen

In [ ]:

# ------------------------------------------------------------
# Adapter
# ------------------------------------------------------------

EMBEDDING_DIM = embeddings.shape[1]

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

loss_fn = nn.TripletMarginWithDistanceLoss(
    distance_function=lambda x, y: (
        1
        - F.cosine_similarity(
            x,
            y,
            dim=-1,
        )
    ),
    margin=MARGIN,
)
adapter = LinearAdapter(embedding_dim=EMBEDDING_DIM).to(DEVICE)


print(adapter)

# Optimizer

In [ ]:
# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    adapter.parameters(),
    lr=LEARNING_RATE,
)


# Sanity Check vor dem Training

In [ ]:
# ------------------------------------------------------------
# Vorher: Similarity Sanity Check
# ------------------------------------------------------------

adapter.eval()


with torch.inference_mode():
    anchor, positive, negative = next(iter(train_loader))

    anchor = anchor.to(DEVICE)
    positive = positive.to(DEVICE)
    negative = negative.to(DEVICE)

    anchor_out = adapter(anchor)
    positive_out = adapter(positive)
    negative_out = adapter(negative)

    positive_similarity = (
        F.cosine_similarity(
            anchor_out,
            positive_out,
            dim=-1,
        )
        .mean()
        .item()
    )

    negative_similarity = (
        F.cosine_similarity(
            anchor_out,
            negative_out,
            dim=-1,
        )
        .mean()
        .item()
    )


print("Before training:")

print(f"Positive similarity: {positive_similarity:.4f}")

print(f"Negative similarity: {negative_similarity:.4f}")


# Training
  
Du solltest jetzt sehen, wie der Loss im Groben sinkt, wobei er nicht zwingend monoton sinken muss.

In [ ]:
# ------------------------------------------------------------
# Training mit Early Stopping auf val-Recall@1
# ------------------------------------------------------------


@torch.inference_mode()
def val_recall_at_1():
    """Recall@1 des Mini-Retrievals auf den val-Sequenzen."""
    adapter.eval()

    db = adapter(torch.from_numpy(val_db_embeddings).float().to(DEVICE))
    qr = adapter(torch.from_numpy(val_q_embeddings).float().to(DEVICE))

    hits = 0
    evaluated = 0
    db_lat = val_db_metadata["lat"].to_numpy()
    db_lon = val_db_metadata["lon"].to_numpy()

    for i in range(len(qr)):
        d = haversine_distance(
            val_q_metadata.iloc[i]["lat"],
            val_q_metadata.iloc[i]["lon"],
            db_lat,
            db_lon,
        )
        ground = np.flatnonzero(d <= UNCERTAIN_RADIUS_M)
        if len(ground) == 0:
            continue
        evaluated += 1
        best = int(torch.argmax(qr[i] @ db.T).item())
        if best in ground:
            hits += 1

    adapter.train()
    return (hits / evaluated if evaluated else float("nan")), evaluated


adapter.train()

best_recall = -1.0
best_state = None
best_epoch = -1
n_val_eval = None

for epoch in range(EPOCHS):
    total_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for anchor, positive, negative in progress:
        anchor = anchor.to(DEVICE)
        positive = positive.to(DEVICE)
        negative = negative.to(DEVICE)

        optimizer.zero_grad()
        loss = loss_fn(adapter(anchor), adapter(positive), adapter(negative))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_loss / len(train_loader)
    recall, n_val_eval = val_recall_at_1()

    marker = ""
    if n_val_eval > 0 and recall > best_recall:
        best_recall = recall
        best_epoch = epoch + 1
        best_state = {
            k: v.detach().cpu().clone() for k, v in adapter.state_dict().items()
        }
        marker = "  <- bestes Modell"

    print(
        f"Epoch {epoch + 1:2d}: loss = {mean_loss:.4f}   val Recall@1 = {recall:.4f}{marker}"
    )


if best_state is None:
    raise RuntimeError(
        "val-Recall war in keiner Epoche auswertbar -- Early Stopping hat nicht "
        "gegriffen und es wuerde das Modell der letzten Epoche gespeichert. "
        "Bitte val-Split pruefen."
    )

adapter.load_state_dict(best_state)

print()
print(f"Bestes Modell aus Epoche {best_epoch} mit val Recall@1 = {best_recall:.4f}")
print(f"(ausgewertet auf {n_val_eval:,} val-Queries mit Abdeckung)")


# Similarity Check nach dem Training
  
                    vorher       nachher

positive similarity   0.XX         ↑
negative similarity   0.XX         ↓

In [ ]:
# ------------------------------------------------------------
# Nachher: Similarity Sanity Check
# ------------------------------------------------------------

adapter.eval()


with torch.inference_mode():
    anchor, positive, negative = next(iter(train_loader))

    anchor = anchor.to(DEVICE)
    positive = positive.to(DEVICE)
    negative = negative.to(DEVICE)

    anchor_out = adapter(anchor)
    positive_out = adapter(positive)
    negative_out = adapter(negative)

    positive_similarity_after = (
        F.cosine_similarity(
            anchor_out,
            positive_out,
            dim=-1,
        )
        .mean()
        .item()
    )

    negative_similarity_after = (
        F.cosine_similarity(
            anchor_out,
            negative_out,
            dim=-1,
        )
        .mean()
        .item()
    )


print("After training:")

print(f"Positive similarity: {positive_similarity_after:.4f}")

print(f"Negative similarity: {negative_similarity_after:.4f}")


# Adapter abspeichern

In [ ]:
# ------------------------------------------------------------
# Adapter speichern
# ------------------------------------------------------------

torch.save(
    adapter.state_dict(),
    ADAPTER_PATH,
)


print("Adapter gespeichert unter:")

print(ADAPTER_PATH)
